# Day 3 — Prompt Engineering + Structured Output Pipeline
---
> **Phase:** 1 — LLM Foundations  
> **Status:** ✅ Complete

## 🧠 Key Concepts

### 1. What Prompt Engineering Actually Is

> **Prompt Engineering = Manipulating the probability distribution of the model's output by shaping its input context.**

The model predicts the next token based on ALL tokens in context.
Better, more precise context → shifts probability toward better tokens.

**The goal is NOT short or long. The goal is precise and unambiguous.**

```
Short + clear role    = good starting point
Short + no context    = ambiguous, model has to guess
Long + clear context  = usually best results
```

### 2. The Six Core Prompt Engineering Techniques

#### Technique 1 — Zero Shot
Ask directly with no examples.
```
Classify this text as positive, negative, or neutral:
"The product arrived late but worked perfectly."
```

#### Technique 2 — Few Shot
Give examples before asking. Model learns the pattern from context (in-context learning).
```
"I loved this product." → positive
"Terrible experience." → negative
"The package arrived." → neutral

Now classify: "The product arrived late but worked perfectly."
```
**Why it works:** Examples become tokens in context. Model sees the pattern and continues it.
This is called **in-context learning** — no retraining needed.

#### Technique 3 — Structured Output Prompting
Force model to return parseable JSON instead of free text.
- Always use `temperature=0` for structured output
- Always specify exact field names and formats
- Always wrap `json.loads()` in try/except

#### Technique 4 — Chain of Thought (CoT)
Force model to show step-by-step reasoning before final answer.
**Why it works:** Reasoning tokens become context for the final answer token.
Better intermediate context → better final answer.
```
# Without CoT — model jumps to answer
"What is 17 x 24?"

# With CoT — model works through steps first
"What is 17 x 24? Show your step by step work."
```

#### Technique 5 — Role Assignment
Give the model a persona via system message.
```python
{"role": "system", "content": "You are an expert Python teacher who explains everything using simple analogies."}
```

#### Technique 6 — Constraint Setting
Explicitly restrict output format, length, vocabulary, or values.
```
sentiment: must be exactly one of: Positive, Negative, Neutral — no other values allowed
```

### 3. Critical Production Lessons

**Label Drift:**
- Model doesn't always return your exact specified values
- Example: returned `"Natural"` instead of `"Neutral"`
- Fix: Be explicit — `"must be exactly one of: Positive, Negative, Neutral"`

**Ambiguous Output Problem:**
- When ground truth is ambiguous, LLM output cannot be trusted blindly
- Solution: Force model to return confidence score
- Use confidence threshold to route to human review vs automated pipeline

**The Confidence Routing Pattern:**
```python
if parsed["confidence"] < 0.85:
    flag_for_human_review()
else:
    pass_to_dashboard()
```

**LLMs will fill ambiguous fields however they want.**
You must be explicit about EVERY field's format in your prompt.

### 4. Prompt vs Code — The Distinction

```
Prompt → what you say to the model (text inside "content" fields)
Code   → Python that constructs messages, sends API call, handles response
```

Writing `"System: ..."` literally in user message text ≠ setting a system message.  
System messages must go in `{"role": "system", "content": "..."}` in code.

## 💻 Code

In [ ]:
# Technique 3 — Structured Output with JSON Parsing
from dotenv import load_dotenv
from groq import Groq
import os
import json

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": """You are a sentiment analysis engine. 
Return only valid raw JSON with no markdown, no backticks, no explanation. 
Nothing except the JSON object with exactly these four fields:
- sentiment: must be exactly one of: Positive, Negative, Neutral
- reasoning: one single sentence explaining the classification
- confidence: float between 0 and 1
- summary: one line summary of the review"""
        },
        {
            "role": "user",
            "content": "The product arrived late but worked perfectly."
        }
    ],
    temperature=0,   # ALWAYS 0 for structured output
    max_tokens=200
)

assistant_reply = response.choices[0].message.content

try:
    parsed = json.loads(assistant_reply)
    print("Sentiment:  ", parsed["sentiment"])
    print("Reasoning:  ", parsed["reasoning"])
    print("Confidence: ", parsed["confidence"])
    print("Summary:    ", parsed["summary"])
    
    if parsed["confidence"] < 0.85:
        print("→ Flag for human review")
    else:
        print("→ Passed to dashboard")
        
except json.JSONDecodeError:
    print("Model returned malformed JSON. Raw output:", assistant_reply)

In [ ]:
# Day 3 Assignment — Full Sentiment Analysis Pipeline
from dotenv import load_dotenv
from groq import Groq
import os
import json

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

conversation_history = [
    {
        "role": "system",
        "content": """You are a sentiment analysis engine. 
Return only valid raw JSON with no markdown, no backticks, no explanation. 
Nothing except the JSON object with exactly these four fields:
- sentiment: must be exactly one of: Positive, Negative, Neutral
- reasoning: one single sentence explaining the classification
- confidence: float between 0 and 1
- summary: one line summary of the review"""
    }
]

print("Sentiment Analysis Pipeline. Type 'quit' to exit.\n")

while True:
    user_input = input("Enter product review: ")
    
    if user_input.lower() == "quit":
        break
    
    conversation_history.append({"role": "user", "content": user_input})
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=conversation_history,
        temperature=0,
        max_tokens=200
    )
    
    assistant_reply = response.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    try:
        parsed = json.loads(assistant_reply)
        print("\nSentiment: ", parsed["sentiment"])
        print("Reasoning: ", parsed["reasoning"])
        print("Confidence:", parsed["confidence"])
        print("Summary:   ", parsed["summary"])
        
        if parsed["confidence"] < 0.85:
            print("→ Flag for human review\n")
        else:
            print("→ Passed to dashboard\n")
            
    except json.JSONDecodeError:
        print("Malformed JSON. Raw output:", assistant_reply)

## 🔬 Observations

| Test | Zero Shot Result | Few Shot Result | Lesson |
|------|-----------------|-----------------|--------|
| Mixed review | Neutral | Positive | Few-shot shifts probability distribution |
| Sarcastic review | ✅ Correctly Negative | — | Model understands sarcasm from context |
| temp=0 structured | Consistent JSON | — | Always use temp=0 for structured output |
| Label drift | `"Natural"` returned | — | Must specify exact allowed values |

## ✅ Day 3 Summary

```
✓ Prompt engineering = manipulating probability distribution via context
✓ Zero-shot prompting
✓ Few-shot prompting and in-context learning
✓ Structured output prompting with JSON parsing
✓ Chain of Thought prompting
✓ Temperature's role in output consistency
✓ Ambiguous outputs need confidence scoring
✓ Label drift — how to prevent it
✓ try/except for robust JSON parsing
✓ Confidence threshold routing pattern
✓ Full sentiment analysis pipeline — built from scratch
```

## 🔜 Coming in Day 4 — Embeddings

**Pre-Day 4 Question (Answered):**
> If you had to search 10,000 product reviews to find ones similar to a query — how would you use vectors?

**Answer:** Convert query to vector → compute similarity (cosine similarity) against all 10,000 review vectors → return most similar ones.

This is the entire foundation of RAG systems.